# Neural Network Training with Metaheuristic Optimizers

Train a simple MLP by optimizing its weights with HPPSO, PSO, PSO-m, or PSO-RIW.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from hppso.algorithms import HPPSO, PSO
from hppso.nn.simple_mlp import SimpleNeuralNetwork, mean_squared_error, nn_objective_function

In [ ]:
data = load_diabetes()
X = StandardScaler().fit_transform(data.data)
y = data.target.reshape(-1, 1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
def train(algo="HPPSO", pop=30, iters=100):
    nn = SimpleNeuralNetwork(X_train.shape[1], 16, 8, 1)
    bounds = [(-2, 2)] * len(nn.get_weights_flat())
    obj = lambda w: nn_objective_function(w, nn, X_train, y_train)

    if algo == "HPPSO":
        opt = HPPSO(obj, n_pop=pop, dimensions=len(bounds), max_it=iters, bounds=(-2, 2))
        score, hist = opt.optimize()
        nn.set_weights_flat(opt.get_best_position())
    else:
        kwargs = {}
        if algo == "PSO-m": kwargs = {"mutation_rate": 0.05, "gaussian_mutation_strength": 0.1}
        if algo == "PSO-RIW": kwargs = {"w_random_range": (0.4, 0.9), "mutation_rate": 0}
        pso = PSO(obj, bounds, pop, iters, **kwargs)
        w, score, hist = pso.optimize()
        nn.set_weights_flat(w)
    test_mse = mean_squared_error(y_test, nn.forward(X_test))
    return score, test_mse, hist

In [ ]:
for algo in ["PSO", "PSO-m", "PSO-RIW", "HPPSO"]:
    train_mse, test_mse, _ = train(algo)
    print(f"{algo:8s} | train MSE={train_mse:.4f} | test MSE={test_mse:.4f}")